# 🧠 Alzheimer's Disease Detection with Deep Learning

This notebook trains a **transfer-learning CNN** (EfficientNet-B3) to classify MRI brain scans into four dementia stages:

| Label | Meaning |
|---|---|
| `No Impairment` | No signs of dementia |
| `Very Mild Impairment` | Very early stage |
| `Mild Impairment` | Mild stage |
| `Moderate Impairment` | Moderate stage |

> ⚠️ **Medical Disclaimer:** This model is a research/educational tool only. It must not be used for clinical diagnosis without review by a qualified medical professional.

---
### Notebook Structure
1. Install & Import Dependencies  
2. Configuration  
3. Data Loading & Augmentation  
4. Model Definition  
5. Training Loop *(with live progress bars)*  
6. Evaluation & Metrics  
7. Single-Image Inference

## 1 · Install & Import Dependencies

In [1]:
# Install required packages (run once)
#%pip install torch torchvision scikit-learn seaborn matplotlib pillow tqdm --user --quiet

Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
from tqdm.notebook import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import os
import time

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print()
    print("⚠️  WARNING: No GPU detected — training will run on CPU and be slow.")
    print("   To fix this, see the 'GPU Setup' cell below before training.")

PyTorch version : 2.10.0+cpu
CUDA available  : False

⚠️  WARNING: No GPU detected — training will run on CPU and be slow.
   To fix this, see the 'GPU Setup' cell below before training.


## 🚀 GPU Setup (Read if CUDA = False)

Your notebook is running on **CPU only**, which makes each epoch take many minutes.
Follow **one** of these options to enable GPU training:

### Option A — Google Colab (free GPU, easiest)
1. Go to **https://colab.research.google.com**
2. Upload this notebook (`File → Upload notebook`)
3. Enable GPU: `Runtime → Change runtime type → T4 GPU → Save`
4. Run all cells — each epoch will take seconds instead of minutes

### Option B — Install CUDA-enabled PyTorch locally
If your PC has an NVIDIA GPU, open **Anaconda Prompt** and run:
```bash
conda install pytorch torchvision pytorch-cuda=11.8 -c pytorch -c nvidia
```
Then restart JupyterLab and re-run. `CUDA available` should now show `True`.

### Option C — Keep CPU but reduce the workload
If you don't have a GPU and can't use Colab, apply the speed settings in the next cell.

## 2 · Configuration

All hyperparameters and paths are here. **CPU-friendly defaults are set** — increase epochs and image size once you have GPU access.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
TRAIN_DIR  = r"C:\Users\cpatterson3\Downloads\actual\Combined Dataset\train"
TEST_DIR   = r"C:\Users\cpatterson3\Downloads\actual\Combined Dataset\test"
MODEL_PATH = "best_model.pth"

# ── Classes (must match your actual folder names exactly) ──────────────────
CLASS_NAMES = ["Mild Impairment", "Moderate Impairment", "No Impairment", "Very Mild Impairment"]
NUM_CLASSES = len(CLASS_NAMES)

# ── Device ─────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ON_CPU = (DEVICE.type == "cpu")

# ── Hyperparameters (auto-scaled for CPU vs GPU) ───────────────────────────
IMAGE_SIZE      = 64  if ON_CPU else 224  # Smaller images = much faster on CPU
BATCH_SIZE      = 64  if ON_CPU else 32   # Larger batches = fewer passes per epoch
EPOCHS          = 10  if ON_CPU else 25   # Keep low on CPU; raise once you have GPU
NUM_WORKERS     = 0   if ON_CPU else 4    # 0 workers avoids Windows multiprocess bugs
LEARNING_RATE   = 1e-4
WEIGHT_DECAY    = 1e-4
DROPOUT_1       = 0.4
DROPOUT_2       = 0.3
FREEZE_BACKBONE = True  # Keep True on CPU — training all layers is very slow

print(f"Device     : {DEVICE}")
print(f"Image size : {IMAGE_SIZE}px")
print(f"Batch size : {BATCH_SIZE}")
print(f"Epochs     : {EPOCHS}")
print(f"Workers    : {NUM_WORKERS}")

## 3 · Data Loading & Augmentation

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(TEST_DIR,  transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=not ON_CPU)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=not ON_CPU)

print(f"Training samples   : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Classes detected   : {train_dataset.classes}")

In [ ]:
# ── Visualise a batch of training samples ──────────────────────────────────
def imshow(tensor, title=None):
    img = tensor.numpy().transpose((1, 2, 0))
    img = np.clip(np.array(IMAGENET_STD) * img + np.array(IMAGENET_MEAN), 0, 1)
    plt.imshow(img)
    if title:
        plt.title(title, fontsize=9)
    plt.axis("off")

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
fig.suptitle("Sample Training Images", fontsize=14, fontweight="bold")
for i, ax in enumerate(axes.flat):
    plt.sca(ax)
    imshow(images[i], title=CLASS_NAMES[labels[i]])
plt.tight_layout()
plt.show()

## 4 · Model Definition

In [ ]:
def build_model(num_classes: int = NUM_CLASSES, freeze_backbone: bool = FREEZE_BACKBONE) -> nn.Module:
    """
    Build an EfficientNet-B3 model with a custom classification head.

    Parameters
    ----------
    num_classes : int
        Number of output classes.
    freeze_backbone : bool
        If True, only the classifier head is trained (much faster on CPU).

    Returns
    -------
    nn.Module
    """
    model = models.efficientnet_b3(weights="IMAGENET1K_V1")

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=DROPOUT_1),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=DROPOUT_2),
        nn.Linear(256, num_classes),
    )
    return model


model = build_model().to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,} / {total:,}")
print(f"(Backbone frozen : {FREEZE_BACKBONE})")

## 5 · Training Loop

Each epoch shows:
- **Outer bar** — overall epoch progress
- **Inner bar** — batch-by-batch progress with live loss
- **Live chart** — loss and accuracy curves updated after every epoch

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0

# ── Live plot setup ────────────────────────────────────────────────────────
%matplotlib inline
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Training Progress (updates each epoch)", fontweight="bold")
plt.tight_layout()
plt.show()

# ── Epoch loop ─────────────────────────────────────────────────────────────
epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Epochs", unit="epoch")

for epoch in epoch_bar:
    start = time.time()

    # ── Training phase ──────────────────────────────────────────────────────
    model.train()
    running_loss, correct = 0.0, 0

    batch_bar = tqdm(
        train_loader,
        desc=f"  Epoch {epoch:02d} train",
        leave=False,
        unit="batch",
    )

    for images, labels in batch_bar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

        # Show live loss in the batch bar
        batch_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    train_acc  = correct / len(train_dataset)

    # ── Validation phase ────────────────────────────────────────────────────
    model.eval()
    val_correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            val_correct += (model(images).argmax(1) == labels).sum().item()

    val_acc = val_correct / len(val_dataset)
    scheduler.step()
    elapsed = time.time() - start

    # ── Record history ──────────────────────────────────────────────────────
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    # ── Save best model ─────────────────────────────────────────────────────
    flag = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        flag = " ✓"

    # ── Update epoch bar ────────────────────────────────────────────────────
    epoch_bar.set_postfix(
        loss=f"{train_loss:.4f}",
        train=f"{train_acc:.3f}",
        val=f"{val_acc:.3f}",
        best=f"{best_val_acc:.3f}",
        secs=f"{elapsed:.0f}s",
    )

    # ── Redraw live chart ────────────────────────────────────────────────────
    epochs_so_far = range(1, epoch + 1)

    ax1.clear()
    ax1.plot(epochs_so_far, history["train_loss"], color="steelblue", label="Train Loss")
    ax1.set_title("Training Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.clear()
    ax2.plot(epochs_so_far, history["train_acc"], color="steelblue", label="Train Acc")
    ax2.plot(epochs_so_far, history["val_acc"],   color="coral",     label="Val Acc")
    ax2.set_title("Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_ylim(0, 1)
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    fig.canvas.draw()
    plt.pause(0.01)

print(f"\n✅ Training complete. Best validation accuracy: {best_val_acc:.4f}")

## 6 · Evaluation & Metrics

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating", leave=False):
        images = images.to(DEVICE)
        preds  = model(images).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
)
plt.title("Confusion Matrix", fontsize=14, fontweight="bold")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

## 7 · Single-Image Inference

In [ ]:
def predict(image_path: str) -> dict:
    """
    Predict the dementia class of a single MRI image.

    Parameters
    ----------
    image_path : str
        Path to a JPEG or PNG brain MRI scan.

    Returns
    -------
    dict
        Class → probability mapping, sorted by descending probability.
    """
    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    image  = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    results = {cls: round(prob.item(), 4) for cls, prob in zip(CLASS_NAMES, probs)}
    return dict(sorted(results.items(), key=lambda x: x[1], reverse=True))


def predict_and_plot(image_path: str) -> None:
    """Display the image alongside a bar chart of class probabilities."""
    probs = predict(image_path)
    predicted_class = next(iter(probs))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Prediction: {predicted_class}", fontsize=13, fontweight="bold")

    ax1.imshow(Image.open(image_path).convert("RGB"), cmap="gray")
    ax1.set_title("Input MRI")
    ax1.axis("off")

    colors = ["coral" if cls == predicted_class else "steelblue" for cls in probs]
    ax2.barh(list(probs.keys()), list(probs.values()), color=colors)
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("Probability")
    ax2.set_title("Class Probabilities")
    ax2.grid(True, axis="x", alpha=0.3)

    plt.tight_layout()
    plt.show()
    print("\nProbabilities:", probs)

In [ ]:
# Replace this path with any MRI image from your dataset
SAMPLE_IMAGE = r"C:\Users\cpatterson3\Downloads\actual\Combined Dataset\test\No Impairment\sample.jpg"

if os.path.exists(SAMPLE_IMAGE):
    predict_and_plot(SAMPLE_IMAGE)
else:
    print(f"Image not found: '{SAMPLE_IMAGE}'")
    print("Update SAMPLE_IMAGE to any .jpg from your test folder and re-run.")